## Function to Evaluate ML Models 

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def evaluate_model(name, model, X_train, y_train, X_test, y_test, binary=True, class_names=None):
    print(f"\n=== {name} ===")

    start_train = time.time()
    model.fit(X_train, y_train)
    end_train = time.time()

    start_pred = time.time()
    y_pred = model.predict(X_test)
    end_pred = time.time()

    acc = accuracy_score(y_test, y_pred)
    train_time = end_train - start_train
    infer_time = end_pred - start_pred

    results = {
        'Model': name,
        'Accuracy': acc,
        'Training Time (s)': train_time,
        'Inference Time (s)': infer_time
    }

    if binary:
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec  = recall_score(y_test, y_pred, zero_division=0)
        f1   = f1_score(y_test, y_pred, zero_division=0)
        far  = fp / (fp + tn) if (fp + tn) > 0 else 0

        results.update({'Precision': prec, 'Recall': rec, 'F1 Score': f1, 'FAR': far})
        labels = ['Benign', 'Attack']
    else:
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        results.update({'Precision (weighted)': prec, 'Recall (weighted)': rec, 'F1 Score (weighted)': f1})

        try:
            y_score = model.predict_proba(X_test)
            auc = roc_auc_score(y_test, y_score, multi_class='ovr', average='weighted')
            results['ROC-AUC (weighted)'] = auc
        except:
            results['ROC-AUC (weighted)'] = None

        labels = list(class_names) if class_names else [f"Class {i}" for i in range(len(np.unique(y_test)))]

    # Print + Confusion Matrix
    print(f"Accuracy: {acc:.4f}")
    print(f"⏱ Training Time: {train_time:.2f} sec")
    print(f"⏱ Inference Time: {infer_time:.2f} sec")
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.title(f"Confusion Matrix — {name}")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    plt.show()

    return results


## Function to Evaluate DL Models 

In [ ]:

def evaluate_dl_model(name, model, X_test, y_test, class_names=None):
    print(f"\n=== {name} ===")

    start_pred = time.time()
    y_pred_raw = model.predict(X_test)
    end_pred = time.time()
    infer_time = end_pred - start_pred

    # Convert predictions
    if y_pred_raw.ndim > 1 and y_pred_raw.shape[1] > 1:
        y_pred = np.argmax(y_pred_raw, axis=1)
        y_true = np.argmax(y_test, axis=1)
        is_binary = False
    else:
        y_pred = (y_pred_raw > 0.5).astype(int).flatten()
        y_true = y_test
        is_binary = True

    acc = accuracy_score(y_true, y_pred)
    results = {
        'Model': name,
        'Accuracy': acc,
        'Training Time (s)': None,
        'Inference Time (s)': infer_time
    }

    if is_binary:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec  = recall_score(y_true, y_pred, zero_division=0)
        f1   = f1_score(y_true, y_pred, zero_division=0)
        far  = fp / (fp + tn) if (fp + tn) > 0 else 0
        results.update({'Precision': prec, 'Recall': rec, 'F1 Score': f1, 'FAR': far})
        labels = ['Benign', 'Attack']
    else:
        prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
        f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        results.update({'Precision (weighted)': prec, 'Recall (weighted)': rec, 'F1 Score (weighted)': f1})

        try:
            auc = roc_auc_score(y_true, y_pred_raw, multi_class='ovr', average='weighted')
            results['ROC-AUC (weighted)'] = auc
        except:
            results['ROC-AUC (weighted)'] = None

        labels = list(class_names) if class_names is not None else [f"Class {i}" for i in range(len(np.unique(y_true)))]

    # Print summary
    print(f"Accuracy:               {acc:.4f}")
    print(f"⏱ Inference Time:       {infer_time:.2f} sec")
    if is_binary:
        print(f"Precision:              {prec:.4f}")
        print(f"Recall:                 {rec:.4f}")
        print(f"F1 Score:               {f1:.4f}")
        print(f"False Alarm Rate (FAR): {far:.4f}")
    else:
        print(f"Precision (weighted):   {prec:.4f}")
        print(f"Recall (weighted):      {rec:.4f}")
        print(f"F1 Score (weighted):    {f1:.4f}")
        if results['ROC-AUC (weighted)'] is not None:
            print(f"ROC-AUC (weighted):  {results['ROC-AUC (weighted)']:.4f}")
        else:
            print("ROC-AUC could not be computed.")

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    # Print confusion matrix as text
    cm_df = pd.DataFrame(cm, index=[f"True {label}" for label in labels],
                              columns=[f"Pred {label}" for label in labels])
    print("\nConfusion Matrix (text view):")
    print(cm_df.to_string())
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.title(f"Confusion Matrix — {name}")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    plt.show()

    return results


## Load Data Sample

In [ ]:
# Load cleaned sample dataset
import pandas as pd

file_path = r'C:\Users\user\CSE-CIC-IDS2018-Full\clean_sample.csv'
df = pd.read_csv(file_path)


## Data Preprocessing

### Multi-class Classification 

In [ ]:
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split


# Encode the 'Label' column
label_encoder = LabelEncoder()
df['Label_encoded'] = label_encoder.fit_transform(df['Label'])

# Split features and target
X = df.drop(columns=['Label', 'Label_encoded'])  # remove original Label
y = df['Label_encoded']

# Stratified train/val/test split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

# Scale features  
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# Final Checks
print("Training set:", X_train_scaled.shape)
print("Validation set:", X_val_scaled.shape)
print("Test set:", X_test_scaled.shape)
print("Number of classes:", len(label_encoder.classes_))
print("Encoded labels:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))


### Binary Classification

In [ ]:

#  Encode the 'Label' column
df['Label_binary'] = df['Label'].apply(lambda x: 0 if x == 'Benign' else 1)

# Split features and target
X_bin = df.drop(columns=['Label', 'Label_encoded', 'Label_binary'])
y_bin = df['Label_binary']

# Stratified train/val/test split 
Xb_train, Xb_temp, yb_train, yb_temp = train_test_split(
    X_bin, y_bin, test_size=0.30, stratify=y_bin, random_state=42)

Xb_val, Xb_test, yb_val, yb_test = train_test_split(
    Xb_temp, yb_temp, test_size=0.5, stratify=yb_temp, random_state=42)

# Scale features 
scaler_bin = StandardScaler()
Xb_train_scaled = scaler_bin.fit_transform(Xb_train)
Xb_val_scaled   = scaler_bin.transform(Xb_val)
Xb_test_scaled  = scaler_bin.transform(Xb_test)

# Final checks
print(" Binary label counts:", y_bin.value_counts().to_dict())
print(" Shapes:", Xb_train_scaled.shape, Xb_test_scaled.shape)


## Train and Evaluate ML Models 

In [ ]:
# Define  models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
}

# Evaluate models for both multiclass and binary classification 
all_results = []

for name, model in models.items():
    print(f"\n Multiclass — {name}")
    result_multi = evaluate_model(
        name=f"{name} (Multiclass)",
        model=model,
        X_train=X_train_scaled,
        y_train=y_train,
        X_test=X_test_scaled,
        y_test=y_test,
        binary=False,
        class_names=label_encoder.classes_
    )
    all_results.append(result_multi)

    print(f"\n Binary — {name}")
    result_bin = evaluate_model(
        name=f"{name} (Binary)",
        model=model,
        X_train=Xb_train_scaled,
        y_train=yb_train,
        X_test=Xb_test_scaled,
        y_test=yb_test,
        binary=True
    )
    all_results.append(result_bin)

# Display final result table
results_df = pd.DataFrame(all_results)
display(results_df)


## Train and Evaluate 1D CNN Model 

### Define and Train the CNN Model (Multiclass Classification)

In [ ]:
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import random
import numpy as np
import tensorflow as tf

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)


# Target one-hot encoding for multiclass classification
y_train_cat = to_categorical(y_train)
y_val_cat = to_categorical(y_val)
y_test_cat = to_categorical(y_test)

# Reshape input for CNN model (flat)
X_train_cnn = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_val_cnn   = X_val_scaled.reshape(X_val_scaled.shape[0], X_val_scaled.shape[1], 1)
X_test_cnn  = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

# Define CNN model
model = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    Conv1D(filters=128, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(y_train_cat.shape[1], activation='softmax')  # number of classes
])

# Compile
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Early stopping to prevent overfitting
es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train model
history = model.fit(
    X_train_cnn, y_train_cat,
    validation_data=(X_val_cnn, y_val_cat),
    epochs=30,
    batch_size=128,
    callbacks=[es],
    verbose=1
)

### Evaluate the CNN Model (Multiclass Classification)

In [ ]:
import matplotlib.pyplot as plt

#Evaluate the CNN model

evaluate_dl_model(
    name="CNN Multiclass",
    model=model,
    X_test=X_test_cnn,
    y_test=y_test_cat,
    class_names=label_encoder.classes_  
)

# Accuracy Plot
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Loss Plot
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


### Define and Train the CNN Model (Binary Classification)

In [ ]:
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# Reshape input
Xb_train_cnn = Xb_train_scaled.reshape(Xb_train_scaled.shape[0], Xb_train_scaled.shape[1], 1)
Xb_val_cnn   = Xb_val_scaled.reshape(Xb_val_scaled.shape[0], Xb_val_scaled.shape[1], 1)
Xb_test_cnn  = Xb_test_scaled.reshape(Xb_test_scaled.shape[0], Xb_test_scaled.shape[1], 1)

model_bin_cnn = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(Xb_train_cnn.shape[1], 1)),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    Conv1D(filters=128, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # Binary classification
])

model_bin_cnn.compile(optimizer='adam',
                      loss='binary_crossentropy',
                      metrics=['accuracy'])
es_bin = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history_bin_cnn = model_bin_cnn.fit(
    Xb_train_cnn, yb_train,
    validation_data=(Xb_val_cnn, yb_val),
    epochs=30,
    batch_size=128,
    callbacks=[es_bin],
    verbose=1
)


### Evaluate the CNN Model (Binary Classification)

In [ ]:
# Evaluate 
results_bin_cnn = evaluate_dl_model(
    name="CNN Binary",
    model=model_bin_cnn,
    X_test=Xb_test_cnn,
    y_test=yb_test,
    class_names=['Benign', 'Attack']
)

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history_bin_cnn.history['accuracy'], label='Train Acc')
plt.plot(history_bin_cnn.history['val_accuracy'], label='Val Acc')
plt.title('CNN Binary Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Loss
plt.subplot(1, 2, 2)
plt.plot(history_bin_cnn.history['loss'], label='Train Loss')
plt.plot(history_bin_cnn.history['val_loss'], label='Val Loss')
plt.title('CNN Binary Model Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()


## Train and Evaluate LSTM Model 

### Define and Train LSTM Model (Multiclass Classification)

In [ ]:
import numpy as np
import tensorflow as tf
import random

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# One-hot encode the targets
y_train_cat = to_categorical(y_train)
y_val_cat   = to_categorical(y_val)
y_test_cat  = to_categorical(y_test)

#Reshape the inputs
X_train_lstm = X_train_scaled.reshape(X_train_scaled.shape[0], 1, X_train_scaled.shape[1])
X_val_lstm   = X_val_scaled.reshape(X_val_scaled.shape[0], 1, X_val_scaled.shape[1])
X_test_lstm  = X_test_scaled.reshape(X_test_scaled.shape[0], 1, X_test_scaled.shape[1])

# Define the LSTM model
model_lstm = Sequential([
    LSTM(128, input_shape=(1, X_train_scaled.shape[1]), return_sequences=False),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(y_train_cat.shape[1], activation='softmax')
])

# Compile model
model_lstm.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Early stopping
es_lstm = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train the model
history_lstm = model_lstm.fit(
    X_train_lstm, y_train_cat,
    validation_data=(X_val_lstm, y_val_cat),
    epochs=30,
    batch_size=128,
    callbacks=[es_lstm],
    verbose=1
)


### Evaluate the LSTM Model (Multiclass Classification)

In [ ]:
#  Evaluate
evaluate_dl_model(
    name="LSTM Multiclass",
    model=model_lstm,
    X_test=X_test_lstm,
    y_test=y_test_cat,
    class_names=label_encoder.classes_  
)

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history_lstm.history['accuracy'], label='Train Accuracy')
plt.plot(history_lstm.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Loss
plt.subplot(1, 2, 2)
plt.plot(history_lstm.history['loss'], label='Train Loss')
plt.plot(history_lstm.history['val_loss'], label='Val Loss')
plt.title('Model Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()


### Define and Train LSTM Model (Binary Classification)

In [ ]:
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# Reshape input
Xb_train_lstm = Xb_train_scaled.reshape(Xb_train_scaled.shape[0], 1, Xb_train_scaled.shape[1])
Xb_val_lstm   = Xb_val_scaled.reshape(Xb_val_scaled.shape[0], 1, Xb_val_scaled.shape[1])
Xb_test_lstm  = Xb_test_scaled.reshape(Xb_test_scaled.shape[0], 1, Xb_test_scaled.shape[1])

model_bin_lstm = Sequential([
    LSTM(64, input_shape=(1, Xb_train_scaled.shape[1]), return_sequences=False),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model_bin_lstm.compile(optimizer='adam',
                       loss='binary_crossentropy',
                       metrics=['accuracy'])

#  Early stopping
es_bin_lstm = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train model 
history_bin_lstm = model_bin_lstm.fit(
    Xb_train_lstm, yb_train,
    validation_data=(Xb_val_lstm, yb_val),
    epochs=15,             
    batch_size=64,         
    callbacks=[es_bin_lstm],
    verbose=1
)


### Evaluate LSTM Model (Binary Classification)

In [ ]:

#  Evaluate 
results_bin_lstm = evaluate_dl_model(
    name="LSTM Binary",
    model=model_bin_lstm,
    X_test=Xb_test_lstm,
    y_test=yb_test,
    class_names=['Benign', 'Attack']
)

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history_bin_lstm.history['accuracy'], label='Train Accuracy')
plt.plot(history_bin_lstm.history['val_accuracy'], label='Val Accuracy')
plt.title('Binary LSTM Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Loss
plt.subplot(1, 2, 2)
plt.plot(history_bin_lstm.history['loss'], label='Train Loss')
plt.plot(history_bin_lstm.history['val_loss'], label='Val Loss')
plt.title('Binary LSTM Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()
